In [1]:
#importss
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("=" * 80)
print("RAINBOW PREDICTION - EXPLORATORY DATA ANALYSIS")
print("=" * 80)

RAINBOW PREDICTION - EXPLORATORY DATA ANALYSIS


In [2]:
# Paths
FEATURES_PATH = Path(r"C:\Rainbow\data\processed\features")
RESULTS_PATH = Path(r"C:\Rainbow\app\results")
FIGURES_PATH = Path(r"C:\Rainbow\app\results\figures")
TABLES_PATH = Path(r"C:\Rainbow\app\results\tables")

# Create directories
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
TABLES_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
# Load full dataset
full_data_file = FEATURES_PATH / "rainbow_data_with_features.csv"
print(f"\nLoading full dataset: {full_data_file}")
df_full = pd.read_csv(full_data_file)
print(f"[DONE] Loaded {len(df_full):,} records with {len(df_full.columns)} features")

# Load high-probability candidates
candidates_file = FEATURES_PATH / "rainbow_favorable_candidates.csv"
print(f"\nLoading rainbow candidates: {candidates_file}")
df_candidates = pd.read_csv(candidates_file)
print(f"[DONE] Loaded {len(df_candidates):,} high-probability records")
print(f"       This is {len(df_candidates)/len(df_full)*100:.2f}% of total data")

# Convert datetime column
df_full['datetime'] = pd.to_datetime(df_full['datetime'])
df_candidates['datetime'] = pd.to_datetime(df_candidates['datetime'])


Loading full dataset: C:\Rainbow\data\processed\features\rainbow_data_with_features.csv
[DONE] Loaded 1,490,832 records with 43 features

Loading rainbow candidates: C:\Rainbow\data\processed\features\rainbow_favorable_candidates.csv
[DONE] Loaded 48,410 high-probability records
       This is 3.25% of total data


In [4]:
# Dataset overview
print("\nFull Dataset:")
print(f"  Total Records: {len(df_full):,}")
print(f"  Date Range: {df_full['datetime'].min()} to {df_full['datetime'].max()}")
print(f"  Cities: {df_full['city'].nunique()}")
print(f"  Total Features: {len(df_full.columns)}")

print("\nRainbow Favorable Candidates:")
print(f"  Total Records: {len(df_candidates):,}")
print(f"  Cities Represented: {df_candidates['city'].nunique()}")
print(f"  Date Range: {df_candidates['datetime'].min()} to {df_candidates['datetime'].max()}")

print("\nRainbow Score Statistics:")
print(df_full['rainbow_score'].describe())


Full Dataset:
  Total Records: 1,490,832
  Date Range: 2020-01-01 00:00:00 to 2024-12-31 23:00:00
  Cities: 34
  Total Features: 43

Rainbow Favorable Candidates:
  Total Records: 48,410
  Cities Represented: 34
  Date Range: 2020-01-01 05:00:00 to 2024-12-31 12:00:00

Rainbow Score Statistics:
count    1.490832e+06
mean     3.709987e+00
std      2.062823e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.000000e+01
Name: rainbow_score, dtype: float64


In [5]:
# Analyze city-level statistics for rainbow potential
city_stats = df_candidates.groupby('city').agg({
    'rainbow_score': ['count', 'mean', 'max'],
    'PRECTOTCORR': 'mean',
    'solar_altitude': 'mean',
    'RH2M': 'mean',
    'T2M': 'mean'
}).round(2)

city_stats.columns = ['Rainbow_Events', 'Avg_Score', 'Max_Score', 
                      'Avg_Precipitation', 'Avg_Solar_Alt', 'Avg_Humidity', 'Avg_Temp']
city_stats = city_stats.sort_values('Rainbow_Events', ascending=False)

print("\nTop 10 Cities by Rainbow Potential:")
print(city_stats.head(10))

# Calculate percentage of favorable conditions per city
city_total = df_full.groupby('city').size()
city_favorable = df_candidates.groupby('city').size()
city_percentage = (city_favorable / city_total * 100).sort_values(ascending=False)

print("\nTop 10 Cities by Percentage of Favorable Conditions:")
for i, (city, pct) in enumerate(city_percentage.head(10).items(), 1):
    print(f"  {i:2d}. {city:25s}: {pct:5.2f}%")

# Save city statistics
city_stats.to_csv(TABLES_PATH / 'city_rainbow_statistics.csv')
print(f"\n[DONE] Saved city statistics to: {TABLES_PATH / 'city_rainbow_statistics.csv'}")



Top 10 Cities by Rainbow Potential:
                    Rainbow_Events  Avg_Score  Max_Score  Avg_Precipitation  \
city                                                                          
Port_Blair                    3023       8.42         10               2.98   
Kavaratti                     2968       8.00          8               2.35   
Puducherry                    2825       8.12         10               2.32   
Thiruvananthapuram            2518       8.00          8               2.29   
Gangtok                       1988       8.32         10               2.95   
Imphal                        1818       8.51         10               3.86   
Aizwal                        1785       8.42         10               4.43   
Chennai                       1759       8.21         10               2.65   
Agartala                      1642       8.41         10               4.06   
Itanagar                      1628       8.43         10               3.59   

              

In [6]:
# Temporal analysis of rainbow conditions
monthly_stats = df_candidates.groupby('month').agg({
    'rainbow_score': ['count', 'mean'],
    'PRECTOTCORR': 'mean',
    'solar_altitude': 'mean'
}).round(2)
monthly_stats.columns = ['Count', 'Avg_Score', 'Avg_Precip', 'Avg_Solar_Alt']

print("\nRainbow Favorable Conditions by Month:")
print(monthly_stats)

# Seasonal analysis
seasonal_stats = df_candidates.groupby('season').agg({
    'rainbow_score': ['count', 'mean'],
    'PRECTOTCORR': 'mean',
    'solar_altitude': 'mean'
}).round(2)
seasonal_stats.columns = ['Count', 'Avg_Score', 'Avg_Precip', 'Avg_Solar_Alt']

print("\nRainbow Favorable Conditions by Season:")
print(seasonal_stats)

# Hourly analysis
hourly_stats = df_candidates.groupby('hour').agg({
    'rainbow_score': ['count', 'mean'],
    'solar_altitude': 'mean'
}).round(2)
hourly_stats.columns = ['Count', 'Avg_Score', 'Avg_Solar_Alt']

print("\nRainbow Favorable Conditions by Hour:")
print(hourly_stats)

# Save temporal statistics
monthly_stats.to_csv(TABLES_PATH / 'monthly_rainbow_statistics.csv')
seasonal_stats.to_csv(TABLES_PATH / 'seasonal_rainbow_statistics.csv')
hourly_stats.to_csv(TABLES_PATH / 'hourly_rainbow_statistics.csv')
print(f"\n[DONE] Saved temporal statistics to: {TABLES_PATH}/")


Rainbow Favorable Conditions by Month:
       Count  Avg_Score  Avg_Precip  Avg_Solar_Alt
month                                             
1       4177       8.46        1.49          28.71
2       1658       8.55        1.77          30.52
3       1160       8.37        2.56          28.05
4       1215       8.18        3.19          23.62
5       1880       8.00        2.89          19.59
6       2857       8.00        3.67          20.87
7       5393       8.00        3.66          20.87
8       7358       8.01        3.18          19.98
9       8104       8.20        3.49          22.47
10      6531       8.42        3.12          23.30
11      3857       8.53        2.28          24.30
12      4220       8.55        1.69          27.28

Rainbow Favorable Conditions by Season:
         Count  Avg_Score  Avg_Precip  Avg_Solar_Alt
season                                              
Autumn   10388       8.46        2.81          23.67
Monsoon  23712       8.07        3.46         

In [7]:
# Create weather categories for analysis
if 'precip_category' not in df_candidates.columns:
    print("\nRecreating weather categories...")
    df_candidates['precip_category'] = pd.cut(df_candidates['PRECTOTCORR'],
                                    bins=[-0.01, 0, 1, 5, np.inf],
                                    labels=['None', 'Light', 'Moderate', 'Heavy'])
    
    df_candidates['temp_category'] = pd.cut(df_candidates['T2M'], 
                                  bins=[-np.inf, 10, 20, 30, np.inf],
                                  labels=['Cold', 'Cool', 'Warm', 'Hot'])
    
    df_candidates['wind_category'] = pd.cut(df_candidates['WS2M'],
                                  bins=[-np.inf, 2, 5, 10, np.inf],
                                  labels=['Calm', 'Light', 'Moderate', 'Strong'])
    print("[DONE] Weather categories created")

print("\nPrecipitation Distribution:")
print(df_candidates['precip_category'].value_counts())

print("\nTemperature Distribution:")
print(df_candidates['temp_category'].value_counts())

print("\nWind Distribution:")
print(df_candidates['wind_category'].value_counts())

print("\nKey Weather Statistics for Favorable Conditions:")
weather_stats = df_candidates[['PRECTOTCORR', 'T2M', 'RH2M', 'WS2M', 
                                'solar_altitude', 'ALLSKY_KT']].describe()
print(weather_stats.round(2))


Precipitation Distribution:
precip_category
Light       20761
Moderate    17569
Heavy       10080
Name: count, dtype: int64

Temperature Distribution:
temp_category
Warm    28500
Hot     11382
Cool     5205
Cold     3323
Name: count, dtype: int64

Wind Distribution:
wind_category
Light       24267
Calm        18933
Moderate     5195
Strong         15
Name: count, dtype: int64

Key Weather Statistics for Favorable Conditions:
       PRECTOTCORR       T2M      RH2M      WS2M  solar_altitude  ALLSKY_KT
count     48410.00  48410.00  48410.00  48410.00        48410.00   48410.00
mean          2.89     25.01     74.11      2.66           23.36     -38.94
std           5.06      9.32      9.30      1.75           12.57     194.63
min           0.01    -38.16     19.55      0.00            0.00    -999.00
25%           0.27     25.13     67.88      1.30           12.45       0.42
50%           1.49     28.22     73.98      2.49           24.96       0.53
75%           4.29     29.87     79.81

In [8]:
# VISUALIZATIONS - CITY ANALYSIS
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: Top 15 cities by rainbow events
top_cities = city_stats.head(15)
axes[0, 0].barh(range(len(top_cities)), top_cities['Rainbow_Events'], color='skyblue')
axes[0, 0].set_yticks(range(len(top_cities)))
axes[0, 0].set_yticklabels(top_cities.index, fontsize=9)
axes[0, 0].set_xlabel('Number of Favorable Events', fontsize=10)
axes[0, 0].set_title('Top 15 Cities by Rainbow Favorable Events', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Plot 2: Top 15 cities by percentage
top_pct_cities = city_percentage.head(15)
axes[0, 1].barh(range(len(top_pct_cities)), top_pct_cities.values, color='lightgreen')
axes[0, 1].set_yticks(range(len(top_pct_cities)))
axes[0, 1].set_yticklabels(top_pct_cities.index, fontsize=9)
axes[0, 1].set_xlabel('Percentage of Favorable Conditions (%)', fontsize=10)
axes[0, 1].set_title('Top 15 Cities by % of Favorable Conditions', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# Plot 3: Average rainbow score by city (top 15)
avg_scores = city_stats.head(15)['Avg_Score'].sort_values(ascending=True)
axes[1, 0].barh(range(len(avg_scores)), avg_scores.values, color='coral')
axes[1, 0].set_yticks(range(len(avg_scores)))
axes[1, 0].set_yticklabels(avg_scores.index, fontsize=9)
axes[1, 0].set_xlabel('Average Rainbow Score', fontsize=10)
axes[1, 0].set_title('Average Rainbow Score - Top 15 Cities', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Plot 4: City comparison - Events vs Average Score
top_20_cities = city_stats.head(20)
axes[1, 1].scatter(top_20_cities['Rainbow_Events'], top_20_cities['Avg_Score'], 
                   s=100, alpha=0.6, c=range(len(top_20_cities)), cmap='viridis')
for idx, city in enumerate(top_20_cities.index):
    axes[1, 1].annotate(city, 
                        (top_20_cities.loc[city, 'Rainbow_Events'], 
                         top_20_cities.loc[city, 'Avg_Score']),
                        fontsize=7, alpha=0.7)
axes[1, 1].set_xlabel('Number of Favorable Events', fontsize=10)
axes[1, 1].set_ylabel('Average Rainbow Score', fontsize=10)
axes[1, 1].set_title('City Comparison: Events vs Score', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'eda_city_analysis.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: eda_city_analysis.png")
plt.close()

[DONE] Saved: eda_city_analysis.png


In [9]:
# VISUALIZATIONS - TEMPORAL ANALYSIS
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: Monthly distribution
axes[0, 0].bar(monthly_stats.index, monthly_stats['Count'], color='steelblue')
axes[0, 0].set_xlabel('Month', fontsize=10)
axes[0, 0].set_ylabel('Number of Favorable Events', fontsize=10)
axes[0, 0].set_title('Rainbow Favorable Events by Month', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Seasonal distribution
season_order = ['Winter', 'Spring', 'Monsoon', 'Autumn']
seasonal_ordered = seasonal_stats.reindex(season_order)
axes[0, 1].bar(range(len(seasonal_ordered)), seasonal_ordered['Count'], 
               color=['#87CEEB', '#90EE90', '#4682B4', '#FF8C00'])
axes[0, 1].set_xticks(range(len(season_order)))
axes[0, 1].set_xticklabels(season_order, fontsize=10)
axes[0, 1].set_ylabel('Number of Favorable Events', fontsize=10)
axes[0, 1].set_title('Rainbow Favorable Events by Season', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Hourly distribution
axes[1, 0].bar(hourly_stats.index, hourly_stats['Count'], color='orange')
axes[1, 0].set_xlabel('Hour of Day', fontsize=10)
axes[1, 0].set_ylabel('Number of Favorable Events', fontsize=10)
axes[1, 0].set_title('Rainbow Favorable Events by Hour', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].axvspan(5, 9, alpha=0.2, color='yellow', label='Early Morning')
axes[1, 0].axvspan(16, 19, alpha=0.2, color='orange', label='Late Afternoon')
axes[1, 0].legend()

# Plot 4: Heatmap - Month vs Hour
month_hour_pivot = df_candidates.groupby(['month', 'hour']).size().unstack(fill_value=0)
sns.heatmap(month_hour_pivot, cmap='YlOrRd', ax=axes[1, 1], cbar_kws={'label': 'Count'})
axes[1, 1].set_xlabel('Hour of Day', fontsize=10)
axes[1, 1].set_ylabel('Month', fontsize=10)
axes[1, 1].set_title('Rainbow Favorable Events: Month vs Hour Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'eda_temporal_patterns.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: eda_temporal_patterns.png")
plt.close()

[DONE] Saved: eda_temporal_patterns.png


In [10]:
# VISUALIZATIONS - WEATHER CONDITIONS
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Precipitation distribution
axes[0, 0].hist(df_candidates['PRECTOTCORR'], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Precipitation (mm/day)', fontsize=10)
axes[0, 0].set_ylabel('Frequency', fontsize=10)
axes[0, 0].set_title('Precipitation Distribution', fontsize=11, fontweight='bold')
axes[0, 0].axvline(df_candidates['PRECTOTCORR'].mean(), color='red', 
                   linestyle='--', label=f'Mean: {df_candidates["PRECTOTCORR"].mean():.2f}')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Temperature distribution
axes[0, 1].hist(df_candidates['T2M'], bins=50, color='red', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Temperature (°C)', fontsize=10)
axes[0, 1].set_ylabel('Frequency', fontsize=10)
axes[0, 1].set_title('Temperature Distribution', fontsize=11, fontweight='bold')
axes[0, 1].axvline(df_candidates['T2M'].mean(), color='blue', 
                   linestyle='--', label=f'Mean: {df_candidates["T2M"].mean():.2f}')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Humidity distribution
axes[0, 2].hist(df_candidates['RH2M'], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[0, 2].set_xlabel('Relative Humidity (%)', fontsize=10)
axes[0, 2].set_ylabel('Frequency', fontsize=10)
axes[0, 2].set_title('Humidity Distribution', fontsize=11, fontweight='bold')
axes[0, 2].axvline(df_candidates['RH2M'].mean(), color='red', 
                   linestyle='--', label=f'Mean: {df_candidates["RH2M"].mean():.2f}')
axes[0, 2].legend()
axes[0, 2].grid(axis='y', alpha=0.3)

# Plot 4: Solar altitude distribution
axes[1, 0].hist(df_candidates['solar_altitude'], bins=50, color='orange', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Solar Altitude (degrees)', fontsize=10)
axes[1, 0].set_ylabel('Frequency', fontsize=10)
axes[1, 0].set_title('Solar Altitude Distribution', fontsize=11, fontweight='bold')
axes[1, 0].axvline(42, color='red', linestyle='--', label='Rainbow Limit (42°)')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 5: Wind speed distribution
axes[1, 1].hist(df_candidates['WS2M'], bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Wind Speed (m/s)', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].set_title('Wind Speed Distribution', fontsize=11, fontweight='bold')
axes[1, 1].axvline(df_candidates['WS2M'].mean(), color='red', 
                   linestyle='--', label=f'Mean: {df_candidates["WS2M"].mean():.2f}')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

# Plot 6: Rainbow score distribution in candidates
axes[1, 2].hist(df_candidates['rainbow_score'], bins=range(8, 12), 
                color='magenta', alpha=0.7, edgecolor='black', align='left')
axes[1, 2].set_xlabel('Rainbow Score', fontsize=10)
axes[1, 2].set_ylabel('Frequency', fontsize=10)
axes[1, 2].set_title('Rainbow Score Distribution (Candidates)', fontsize=11, fontweight='bold')
axes[1, 2].set_xticks([8, 9, 10])
axes[1, 2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'eda_weather_conditions.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: eda_weather_conditions.png")
plt.close()

[DONE] Saved: eda_weather_conditions.png


In [11]:
# Select key features for correlation
correlation_features = [
    'rainbow_score', 'solar_altitude', 'PRECTOTCORR', 'RH2M', 'T2M', 
    'WS2M', 'ALLSKY_KT', 'hour', 'month', 'day_of_year'
]

corr_matrix = df_candidates[correlation_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix - Rainbow Favorable Conditions', 
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'eda_correlation_matrix.png', dpi=300, bbox_inches='tight')
print("[DONE] Saved: eda_correlation_matrix.png")
plt.close()

[DONE] Saved: eda_correlation_matrix.png


In [12]:
# Extract key insights for summary
top_city = city_stats.index[0]
top_city_events = city_stats.iloc[0]['Rainbow_Events']
top_month = monthly_stats.idxmax()['Count']
top_season = seasonal_stats.idxmax()['Count']
peak_hour = hourly_stats.idxmax()['Count']

insights = f"""
EDA KEY INSIGHTS - RAINBOW PREDICTION
{'=' * 80}

1. GEOGRAPHICAL INSIGHTS:
   - Top City: {top_city} with {top_city_events:.0f} favorable events
   - Cities with highest percentage: {', '.join(city_percentage.head(3).index.tolist())}
   - Total cities with favorable conditions: {df_candidates['city'].nunique()}

2. TEMPORAL INSIGHTS:
   - Best Season: {top_season} (avg score: {seasonal_stats.loc[top_season, 'Avg_Score']:.2f})
   - Best Month: Month {top_month} with {monthly_stats.loc[top_month, 'Count']:.0f} events
   - Peak Hour: {peak_hour}:00 with {hourly_stats.loc[peak_hour, 'Count']:.0f} events
   - Early Morning (5-9 AM): {df_candidates[df_candidates['hour'].between(5, 9)].shape[0]:,} events
   - Late Afternoon (16-19): {df_candidates[df_candidates['hour'].between(16, 19)].shape[0]:,} events

3. WEATHER CONDITIONS:
   - Average Precipitation: {df_candidates['PRECTOTCORR'].mean():.2f} mm/day
   - Average Temperature: {df_candidates['T2M'].mean():.2f}°C
   - Average Humidity: {df_candidates['RH2M'].mean():.2f}%
   - Average Solar Altitude: {df_candidates['solar_altitude'].mean():.2f}°
   - Average Wind Speed: {df_candidates['WS2M'].mean():.2f} m/s

4. RAINBOW SCORE DISTRIBUTION:
   - Score 8: {df_candidates[df_candidates['rainbow_score'] == 8].shape[0]:,} events
   - Score 9: {df_candidates[df_candidates['rainbow_score'] == 9].shape[0]:,} events
   - Score 10: {df_candidates[df_candidates['rainbow_score'] == 10].shape[0]:,} events

5. LABELING STRATEGY RECOMMENDATIONS:
   - Focus on top 5 cities: {', '.join(city_stats.head(5).index.tolist())}
   - Target months: {', '.join([f'Month {m}' for m in monthly_stats.nlargest(3, 'Count').index.tolist()])}
   - Target hours: {', '.join([f'{h}:00' for h in hourly_stats.nlargest(3, 'Count').index.tolist()])}
   - This covers {(city_stats.head(5)['Rainbow_Events'].sum() / len(df_candidates) * 100):.1f}% of all favorable events

{'=' * 80}
"""

print(insights)

# Save insights
insights_file = RESULTS_PATH / 'eda_insights_summary.txt'
with open(insights_file, 'w', encoding='utf-8') as f:
    f.write(insights)
print(f"\n[DONE] Saved insights to: {insights_file}")


EDA KEY INSIGHTS - RAINBOW PREDICTION

1. GEOGRAPHICAL INSIGHTS:
   - Top City: Port_Blair with 3023 favorable events
   - Cities with highest percentage: Port_Blair, Kavaratti, Puducherry
   - Total cities with favorable conditions: 34

2. TEMPORAL INSIGHTS:
   - Best Season: Monsoon (avg score: 8.07)
   - Best Month: Month 9 with 8104 events
   - Peak Hour: 11:00 with 12695 events
   - Early Morning (5-9 AM): 11,466 events
   - Late Afternoon (16-19): 0 events

3. WEATHER CONDITIONS:
   - Average Precipitation: 2.89 mm/day
   - Average Temperature: 25.01°C
   - Average Humidity: 74.11%
   - Average Solar Altitude: 23.36°
   - Average Wind Speed: 2.66 m/s

4. RAINBOW SCORE DISTRIBUTION:
   - Score 8: 41,666 events
   - Score 9: 1,175 events
   - Score 10: 5,569 events

5. LABELING STRATEGY RECOMMENDATIONS:
   - Focus on top 5 cities: Port_Blair, Kavaratti, Puducherry, Thiruvananthapuram, Gangtok
   - Target months: Month 9, Month 8, Month 10
   - Target hours: 11:00, 12:00, 10:00
   

In [13]:
print("\nFiles Created:")
print(f"1. City statistics: {TABLES_PATH / 'city_rainbow_statistics.csv'}")
print(f"2. Monthly statistics: {TABLES_PATH / 'monthly_rainbow_statistics.csv'}")
print(f"3. Seasonal statistics: {TABLES_PATH / 'seasonal_rainbow_statistics.csv'}")
print(f"4. Hourly statistics: {TABLES_PATH / 'hourly_rainbow_statistics.csv'}")
print(f"5. City analysis visualization: {FIGURES_PATH / 'eda_city_analysis.png'}")
print(f"6. Temporal patterns visualization: {FIGURES_PATH / 'eda_temporal_patterns.png'}")
print(f"7. Weather conditions visualization: {FIGURES_PATH / 'eda_weather_conditions.png'}")
print(f"8. Correlation matrix: {FIGURES_PATH / 'eda_correlation_matrix.png'}")
print(f"9. Insights summary: {insights_file}")

print("\n" + "=" * 80)


Files Created:
1. City statistics: C:\Rainbow\app\results\tables\city_rainbow_statistics.csv
2. Monthly statistics: C:\Rainbow\app\results\tables\monthly_rainbow_statistics.csv
3. Seasonal statistics: C:\Rainbow\app\results\tables\seasonal_rainbow_statistics.csv
4. Hourly statistics: C:\Rainbow\app\results\tables\hourly_rainbow_statistics.csv
5. City analysis visualization: C:\Rainbow\app\results\figures\eda_city_analysis.png
6. Temporal patterns visualization: C:\Rainbow\app\results\figures\eda_temporal_patterns.png
7. Weather conditions visualization: C:\Rainbow\app\results\figures\eda_weather_conditions.png
8. Correlation matrix: C:\Rainbow\app\results\figures\eda_correlation_matrix.png
9. Insights summary: C:\Rainbow\app\results\eda_insights_summary.txt

